[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/module_08_defenses/Lab_8b_Federated_Learning_and_Adversarial_Training.ipynb)

# Lab 8b: Federated Learning with Medical Data

**Course:** Introduction to Data Security  
**Module 8:** Defenses  
**Estimated Time:** 90 minutes

---

<div align="center">
  <img src="https://raw.githubusercontent.com/ELTE-DSED/Intro-Data-Security/main/module_08_defenses/images/federation_figure.jpg" width=60% alt="federation figure">
</div>

This lab shows how hospitals can collaborate on a shared medical-imaging task without pooling raw patient data.



Medical data is sensitive, so hospitals should not centralize raw patient records if collaboration can be achieved in a privacy-preserving way. In this notebook, we use a public chest X-ray benchmark to model that setting and show how federated learning reduces data exposure.

### Why Federated Learning for Healthcare?

- **Privacy**: Raw patient data never leaves the hospital.
- **Regulation**: Complies with GDPR, HIPAA, and other data protection laws.
- **Collaboration**: Hospitals can benefit from shared learning without exposing individual records.

### Before we start

Before we start, please run the following to make sure that your environment is correctly setup.

In [ ]:
%pip install --quiet flwr medmnist

### Dependencies

In [ ]:
import flwr as fl
import medmnist
from medmnist import INFO
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict, Counter
import random

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

### Preparing the input data

Loading PneumoniaMNIST (public chest X-ray dataset)


In [ ]:

DataClass = getattr(medmnist, INFO['pneumoniamnist']['python_class'])
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_dataset = DataClass(split='train', transform=transform, download=True)
test_dataset = DataClass(split='test', transform=transform, download=True)

train_data = train_dataset
test_data = test_dataset

test_loader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=2)

sample_x, _ = train_data[0]
input_shape = tuple(sample_x.shape)

label_meta = INFO['pneumoniamnist'].get('label', None)
if isinstance(label_meta, dict):
    ordered_keys = sorted(label_meta.keys(), key=lambda k: int(k) if str(k).isdigit() else str(k))
    class_names = [str(label_meta[k]) for k in ordered_keys]
elif isinstance(label_meta, list):
    class_names = [str(x) for x in label_meta]
else:
    class_names = []

num_classes = int(np.unique(np.array(train_dataset.labels).reshape(-1)).size)
class_desc = " / ".join(class_names) if class_names else "unknown"

print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")
print(f"Input shape: {input_shape} - grayscale chest X-ray images")
print(f"Classes: {num_classes} ({class_desc})")

# Gather labels from the training set
train_labels = [int(np.array(train_data[i][1]).squeeze()) for i in range(len(train_data))]
counts = Counter(train_labels)
# Plot class distribution
plt.figure(figsize=(6, 4))
labels_sorted = sorted(counts.keys())
label_names = [class_names[i] if i < len(class_names) else str(i) for i in labels_sorted]
plt.bar(label_names, [counts[i] for i in labels_sorted], color='#2ecc71')
plt.title('Training set class distribution')
plt.ylabel('Count')
plt.xlabel('Class')
plt.tight_layout()
plt.show()

In [ ]:
# Show one example image per class (unnormalized for display)
num_to_show = max(1, num_classes)
fig, axes = plt.subplots(1, num_to_show, figsize=(3 * num_to_show, 3))
if num_to_show == 1:
    axes = [axes]
for ax_idx, cls in enumerate(range(num_to_show)):
    # find a random index for this class
    indices_cls = [i for i, lbl in enumerate(train_labels) if lbl == cls]
    if len(indices_cls) == 0:
        axes[ax_idx].axis('off')
        continue
    idx = random.choice(indices_cls)
    img, lbl = train_data[idx]
    img = img.squeeze().numpy()
    # Un-normalize (was normalized with mean=0.5, std=0.5)
    img_disp = img * 0.5 + 0.5
    axes[ax_idx].imshow(img_disp, cmap='gray')
    title = (class_names[cls] if cls < len(class_names) else str(cls)) + f'\nLabel={int(np.array(lbl).squeeze())}'
    axes[ax_idx].set_title(title)
    axes[ax_idx].axis('off')
plt.tight_layout()
plt.show()

### Model Definition

In [ ]:
class Net(nn.Module):
    def __init__(self, num_classes=2):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            target = target.to(device).long().view(-1)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * correct / total

## Flower-Based Federated Learning (FedAvg) <a id="fl"></a>

Using Flower to coordinate a simple cross-silo federated-learning workflow for hospitals. Each client trains locally on its own partition, then the server aggregates the model updates.

In [ ]:
def partition(dataset, n_clients=5, non_iid=False):
    """Split the medical dataset into client partitions representing hospitals."""
    indices = np.arange(len(dataset))
    labels = np.array([int(np.array(dataset[i][1]).squeeze()) for i in indices])

    if non_iid:
        sorted_indices = indices[np.argsort(labels)]
        client_splits = np.array_split(sorted_indices, n_clients)
    else:
        rng = np.random.default_rng(42)
        client_buckets = [[] for _ in range(n_clients)]
        for label in np.unique(labels):
            class_indices = indices[labels == label].copy()
            rng.shuffle(class_indices)
            for position, sample_idx in enumerate(class_indices):
                client_buckets[position % n_clients].append(int(sample_idx))
        client_splits = [np.array(split) for split in client_buckets]

    return [Subset(dataset, split.tolist()) for split in client_splits]

In [ ]:
def get_parameters(model):
    return [val.detach().cpu().numpy() for _, val in model.state_dict().items()]

def set_parameters(model, parameters):
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)

In [ ]:
def local_train(model, loader, epochs=1, lr=0.01):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for data, target in loader:
            data = data.to(device)
            target = target.to(device).long().view(-1)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

In [ ]:
class Client:
    def __init__(self, model, train_loader, eval_loader):
        self.model = model
        self.train_loader = train_loader
        self.eval_loader = eval_loader

    def get_parameters(self):
        return get_parameters(self.model)

    def fit(self, parameters, config):
        set_parameters(self.model, parameters)
        local_epochs = int(config.get("local_epochs", 1))
        local_train(self.model, self.train_loader, epochs=local_epochs)
        return get_parameters(self.model), len(self.train_loader.dataset), {}

    def evaluate(self, parameters):
        set_parameters(self.model, parameters)
        accuracy = evaluate(self.model, self.eval_loader)
        return 0.0, len(self.eval_loader.dataset), {"accuracy": accuracy}

In [ ]:
def run_fedavg(global_model, client_loaders, rounds=5):
    """Run a simple Flower-style FedAvg loop across hospital partitions."""
    global_parameters = get_parameters(global_model)
    global_accuracies = []

    for r in range(rounds):
        client_results = []

        for loader in client_loaders:
            client_model = Net().to(device)
            set_parameters(client_model, global_parameters)
            client = Client(client_model, loader, loader)
            parameters_prime, num_examples, _ = client.fit(global_parameters, {"local_epochs": 2})
            client_results.append((num_examples, parameters_prime))

        total_examples = sum(num_examples for num_examples, _ in client_results)
        global_parameters = [
            sum(num_examples * client_params[i] for num_examples, client_params in client_results) / total_examples
            for i in range(len(client_results[0][1]))
        ]

        set_parameters(global_model, global_parameters)
        acc = evaluate(global_model, test_loader)
        global_accuracies.append(acc)
        print(f"Round {r + 1}/{rounds}: Global Test Accuracy = {acc:.2f}%")

    return global_model, global_accuracies

In [ ]:
# Create hospital-style data partitions for Flower clients
clients = partition(train_data, n_clients=5, non_iid=False)
client_loaders = [DataLoader(c, batch_size=64, shuffle=True, num_workers=2) for c in clients]

print(f"Created {len(clients)} hospital clients with balanced partitions.")

# Flower-based Federated Averaging training loop
global_model = Net().to(device)
rounds = 8
global_model, global_accuracies = run_fedavg(global_model, client_loaders, rounds=rounds)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(range(1, rounds + 1), global_accuracies, marker='o', linewidth=2.5,
        markersize=8, color='#3498db', label='Flower FedAvg (Balanced partitions)')
ax.set_xlabel('Federated Round', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Flower FedAvg Convergence on PneumoniaMNIST', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, linestyle='--')
ax.legend(fontsize=11)
ax.set_ylim((max(0, min(global_accuracies) - 2), 100))
plt.tight_layout()
plt.show()